In [ ]:
from pathlib import Path
import shutil
from tqdm import tqdm
import json
import re

# Пути
input_root = Path("Digital_core_v4")
output_root = Path("Digital_core_v5")

# Очищаем старую папку
if output_root.exists():
    print("Очистка старой папки...")
    shutil.rmtree(output_root)
output_root.mkdir(exist_ok=True)

# Маппинг
class_mapping = {
    'Песчаник': 'Песчаник',
    'Песчаник карбонатный': 'Песчаник',
    'Песчаник с прослоями алевролита': 'Песчаник',
    'Песчаник с прослоями аргиллита': 'Песчаник',
    'Аргиллит': 'Аргиллит',
    'Алевролит': 'Алевролит',
    'Переслаивание': 'Переслаивание',
    'Глина': 'Глина',
    'Уголь': 'Углистые породы',
    'Уголь с прослоями аргиллита': 'Углистые породы',
    'Известняк': 'Карбонатные породы',
    'Глинисто-карбонатная порода': 'Карбонатные породы',
    'Породы фундамента': 'Породы фундамента',
}

def extract_mineral(filename):
    name = filename.stem
    name = re.sub(r'(_frag\d+|_part\d+)', '', name)
    match = re.search(r'(\d+\.?\d*)', name)
    if match:
        mineral = name[:match.start()].strip('_')
    else:
        mineral = name
    return mineral

print("=" * 60)
print("СОЗДАНИЕ DATASET_DS И DATASET_UV В DIGITAL_CORE_V5")
print("=" * 60)

for light in ["ДС", "УФ"]:
    print(f"\n📁 Обработка {light}...")
    
    light_output = output_root / light
    light_output.mkdir(exist_ok=True)
    
    # Сначала собираем все файлы
    all_files = []
    for field_dir in input_root.iterdir():
        if not field_dir.is_dir():
            continue
        light_input = field_dir / light
        if not light_input.exists():
            continue
        for img_path in light_input.glob("*.jpg"):
            all_files.append(img_path)
    
    print(f"  Найдено файлов: {len(all_files)}")
    
    class_counts = {}
    
    # Копируем с прогресс-баром
    for img_path in tqdm(all_files, desc=f"  Копирование {light}"):
        mineral_raw = extract_mineral(img_path)
        class_name = None
        for key, value in class_mapping.items():
            if key in mineral_raw:
                class_name = value
                break
        if class_name is None:
            class_name = "Прочие"
        
        class_dir = light_output / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        
        dst = class_dir / img_path.name
        if not dst.exists():
            shutil.copy2(img_path, dst)
            class_counts[class_name] = class_counts.get(class_name, 0) + 1
    
    print(f"\n  {light}: распределение по классам:")
    for name, cnt in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"    {name}: {cnt}")

print("\n✅ ГОТОВО")

# Финальная проверка
print("\n" + "=" * 60)
print("ИТОГОВАЯ СТРУКТУРА:")
print("=" * 60)
for light in ["ДС", "УФ"]:
    light_dir = output_root / light
    if light_dir.exists():
        total = sum(len(list(d.glob("*.jpg"))) for d in light_dir.iterdir() if d.is_dir())
        print(f"  {light}: {total} файлов")
        for class_dir in light_dir.iterdir():
            if class_dir.is_dir():
                cnt = len(list(class_dir.glob("*.jpg")))
                print(f"    {class_dir.name}: {cnt}")

In [ ]:
from pathlib import Path
from collections import defaultdict

ds_dir = Path("Digital_core_v5/ДС")
uv_dir = Path("Digital_core_v5/УФ")

print("=" * 60)
print("ПРОВЕРКА БАЛАНСА КЛАССОВ")
print("=" * 60)

def get_class_stats(light_dir, light_name):
    if not light_dir.exists():
        print(f"{light_name}: папка не найдена")
        return {}
    
    stats = {}
    for class_dir in light_dir.iterdir():
        if class_dir.is_dir():
            count = len(list(class_dir.glob("*.jpg")))
            stats[class_dir.name] = count
    return stats

ds_stats = get_class_stats(ds_dir, "ДС")
uv_stats = get_class_stats(uv_dir, "УФ")

print("\n📊 ДС (дневной свет):")
for cls, cnt in sorted(ds_stats.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * (cnt // 1000)
    print(f"  {cls:30s}: {cnt:6d} {bar}")

print("\n📊 УФ (ультрафиолет):")
for cls, cnt in sorted(uv_stats.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * (cnt // 1000)
    print(f"  {cls:30s}: {cnt:6d} {bar}")

print("\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА:")
print("=" * 60)
all_classes = set(ds_stats.keys()) | set(uv_stats.keys())
print(f"{'Класс':30s} {'ДС':>8s} {'УФ':>8s} {'Всего':>8s}")
print("-" * 55)
for cls in sorted(all_classes):
    ds = ds_stats.get(cls, 0)
    uv = uv_stats.get(cls, 0)
    total = ds + uv
    print(f"{cls:30s} {ds:8d} {uv:8d} {total:8d}")

In [ ]:
from pathlib import Path
import shutil

ds_dir = Path("Digital_core_v5/ДС")
uv_dir = Path("Digital_core_v5/УФ")

# Удаляем папки Карбонатные породы
for light_dir in [ds_dir, uv_dir]:
    if not light_dir.exists():
        continue
    
    carbon_dir = light_dir / "Карбонатные породы"
    if carbon_dir.exists():
        shutil.rmtree(carbon_dir)
        print(f"🗑️ Удалено: {carbon_dir}")

print("\n✅ Карбонатные породы исключены")

# Проверка оставшихся классов
print("\n" + "=" * 60)
print("ОСТАВШИЕСЯ КЛАССЫ")
print("=" * 60)

for light_name, light_dir in [("ДС", ds_dir), ("УФ", uv_dir)]:
    print(f"\n{light_name}:")
    for class_dir in light_dir.iterdir():
        if class_dir.is_dir():
            count = len(list(class_dir.glob("*.jpg")))
            print(f"  {class_dir.name}: {count}")

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import random
import shutil
import time
from tqdm import tqdm
from PIL import Image

ds_dir = Path("Digital_core_v5/ДС")
uv_dir = Path("Digital_core_v5/УФ")
backup_dir = Path("Digital_core_v5_backup")

TARGET = 5000
classes_to_augment = ["Углистые породы", "Прочие"]

def load_image(path):
    with Image.open(path) as img:
        return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def augment_image(img):
    img = img.copy()
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
    if random.random() > 0.5:
        alpha = 0.8 + random.random() * 0.4
        img = cv2.convertScaleAbs(img, alpha=alpha)
    if random.random() > 0.5:
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    if random.random() > 0.5:
        h, w = img.shape[:2]
        shift = int(w * 0.03 * (random.random() - 0.5))
        M = np.float32([[1, 0, shift], [0, 1, 0]])
        img = cv2.warpAffine(img, M, (w, h))
    return img

print("=" * 60)
print("АУГМЕНТАЦИЯ МАЛЕНЬКИХ КЛАССОВ")
print("=" * 60)

# Создаём бэкап
if not backup_dir.exists():
    print("\n📦 Создание бэкапа...")
    shutil.copytree("Digital_core_v5", str(backup_dir))
    print("   ✅ Бэкап создан")

for light_name, light_dir in [("ДС", ds_dir), ("УФ", uv_dir)]:
    if not light_dir.exists():
        continue
    
    print(f"\n📁 {light_name}:")
    for class_name in classes_to_augment:
        class_path = light_dir / class_name
        if not class_path.exists():
            continue
        
        current_files = [f for f in class_path.glob("*.jpg") if "_aug_" not in f.name]
        current_count = len(current_files)
        
        if current_count >= TARGET:
            print(f"  {class_name}: уже {current_count} (>= {TARGET})")
            continue
        
        needed = TARGET - current_count
        print(f"  {class_name}: {current_count} → нужно +{needed}")
        
        created = 0
        start_time = time.time()
        
        pbar = tqdm(total=needed, desc=f"    Создание {class_name}", unit="файл")
        
        while created < needed:
            for src_path in current_files:
                if created >= needed:
                    break
                try:
                    img = load_image(src_path)
                    aug_img = augment_image(img)
                    new_name = f"{src_path.stem}_aug_{created:04d}.jpg"
                    Image.fromarray(cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB)).save(str(class_path / new_name))
                    created += 1
                    pbar.update(1)
                except Exception as e:
                    print(f"      Ошибка: {src_path.name} - {e}")
                    continue
        
        pbar.close()
        elapsed = time.time() - start_time
        final_total = len(list(class_path.glob("*.jpg")))
        print(f"    ✅ За {elapsed:.1f} сек: всего {final_total} файлов")

print("\n✅ АУГМЕНТАЦИЯ ЗАВЕРШЕНА")

In [ ]:
import random
import shutil
from pathlib import Path

ds_dir = Path("Digital_core_v5/ДС")
uv_dir = Path("Digital_core_v5/УФ")

output_root = Path("Digital_core_v5_split")
output_root.mkdir(exist_ok=True)

manual_dir = Path("Digital_core_v5_manual")
manual_dir.mkdir(exist_ok=True)

TRAIN_RATIO = 0.8
VAL_RATIO = 0.2

random.seed(42)

def split_files(files):
    random.shuffle(files)
    n = len(files)
    train_end = int(n * TRAIN_RATIO)
    train = files[:train_end]
    val = files[train_end:]
    return train, val

def copy_files(file_list, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    for f in file_list:
        shutil.copy2(f, dest_dir / f.name)

print("=" * 60)
print("РАЗДЕЛЕНИЕ НА TRAIN/VAL (80/20)")
print("=" * 60)

for light_name, light_dir in [("ДС", ds_dir), ("УФ", uv_dir)]:
    if not light_dir.exists():
        continue
    
    print(f"\n📁 {light_name}:")
    
    for class_dir in light_dir.iterdir():
        if not class_dir.is_dir():
            continue
        
        files = list(class_dir.glob("*.jpg"))
        if not files:
            continue
        
        train, val = split_files(files)
        
        train_dir = output_root / light_name / "train" / class_dir.name
        val_dir = output_root / light_name / "val" / class_dir.name
        
        copy_files(train, train_dir)
        copy_files(val, val_dir)
        
        print(f"  {class_dir.name}: train={len(train)}, val={len(val)}")
        
        # Отдельная папка: 2 случайных фото из val (модель их не видела)
        if len(val) >= 2:
            manual_class_dir = manual_dir / light_name / class_dir.name
            manual_class_dir.mkdir(parents=True, exist_ok=True)
            for f in random.sample(val, 2):
                shutil.copy2(f, manual_class_dir / f.name)
            print(f"      → 2 фото из val (модель не видела) скопированы в {manual_class_dir}")

print("\n" + "=" * 60)
print("ГОТОВО!")
print(f"  Train/Val датасет: {output_root}")
print(f"  Для ручной проверки (модель не видела): {manual_dir}")
print("=" * 60)